In [2]:
from qdrant_client import QdrantClient

# LOCAL Qdrant DB created by your pipeline
qdrant_client = QdrantClient(path="paper_qdrant_db")

all_names = [c.name for c in qdrant_client.get_collections().collections]
paper_names = [name for name in all_names if name.startswith("paper_")]

print("All collections:")
for name in all_names:
    print(" -", name)

print("\nPaper collections to delete:")
for name in paper_names:
    print(" -", name)

for name in paper_names:
    qdrant_client.delete_collection(collection_name=name)
    print("Deleted:", name)

print(f"\nDeleted {len(paper_names)} paper collection(s).")

All collections:

Paper collections to delete:

Deleted 0 paper collection(s).


In [3]:
from __future__ import annotations

import datetime
import json
import re
import shlex
import shutil
import subprocess
import textwrap
import time
from pathlib import Path


# =========================================================
# SETTINGS
# =========================================================
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"
# MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

EMBEDDING_MODEL_ID = "BAAI/bge-large-en-v1.5"
EMBEDDING_DIM = 1024

WALLTIME = "04:45:00"
MAX_WAIT_MINUTES = 330
POLL_SECONDS = 20
GPU_REQUEST = 2
HOST_PREDICATE = "network_address like 'larochette-%'"

TORCH_INDEX_URL = "https://download.pytorch.org/whl/rocm6.1"
TORCH_VERSION = "2.5.1"
TORCHVISION_VERSION = "0.20.1"
TORCHAUDIO_VERSION = "2.5.1"
HSA_OVERRIDE_GFX_VERSION = "9.0.10"

SSH_HOST = "lux"

# =========================================================
# LOCAL JDH LAYOUT
# =========================================================
JDH_ROOT = Path("./")

PRIMARY_PAPER_FILES_ROOT = JDH_ROOT / "review_outputs" / "paper_reviews"
FALLBACK_PAPER_FILES_ROOT = JDH_ROOT / "paper_reviews"
PAPER_FILES_ROOT_CANDIDATES = [
    PRIMARY_PAPER_FILES_ROOT,
    FALLBACK_PAPER_FILES_ROOT,
]

LOCAL_RESULTS_ROOT = JDH_ROOT / "g5k_paper_chunk_results"
LOCAL_COLLECTIONS_DIR = JDH_ROOT / "review_outputs" / "paper_collections"
LOCAL_QDRANT_DIR = JDH_ROOT / "review_outputs" / "paper_qdrant_db"

QDRANT_MODE = "local"   # "local" or "server"
QDRANT_URL = ""
QDRANT_API_KEY = ""


# =========================================================
# GRID5000 CLIENT
# =========================================================
class Grid5000Client:
    def __init__(self, ssh_host: str = SSH_HOST, local_results_root: Path | str = LOCAL_RESULTS_ROOT):
        self.ssh_host = ssh_host
        self.local_results_root = Path(local_results_root)
        self.local_results_root.mkdir(parents=True, exist_ok=True)

    def run(self, cmd: str, input_text: str | None = None) -> str:
        cmd = textwrap.dedent(cmd).strip()
        if not cmd:
            return ""

        remote = "bash -lc " + shlex.quote(cmd)
        p = subprocess.run(
            ["ssh", self.ssh_host, remote],
            input=input_text,
            text=True,
            capture_output=True,
        )
        if p.returncode != 0:
            raise RuntimeError(
                f"Remote cmd failed (rc={p.returncode}).\n"
                f"--- CMD ---\n{cmd}\n"
                f"--- STDERR ---\n{p.stderr}"
            )
        return p.stdout

    def run_ok(self, cmd: str) -> str:
        try:
            return self.run(cmd)
        except Exception:
            return ""

    def remote_exists(self, path: str) -> bool:
        out = self.run_ok(f'test -e {shlex.quote(path)} && echo YES || true').strip()
        return out == "YES"

    def remote_exists_nonempty(self, path: str) -> bool:
        out = self.run_ok(f'test -s {shlex.quote(path)} && echo YES || true').strip()
        return out == "YES"

    def read_remote_text(self, path: str) -> str:
        return self.run_ok(f"cat {shlex.quote(path)} 2>/dev/null || true")

    def prepare_home(self) -> str:
        return self.run(r"""
            set -euo pipefail

            USERNAME="$(id -un)"
            HOME_DIR="$HOME"

            echo "[CLEANUP] removing old launcher/model/cache artifacts from HOME and /tmp"

            rm -rf "$HOME_DIR/.qa_launcher" 2>/dev/null || true
            mkdir -p "$HOME_DIR/.qa_launcher"

            rm -rf "$HOME_DIR/model-store" 2>/dev/null || true
            rm -rf "$HOME_DIR/.cache/huggingface" 2>/dev/null || true
            rm -rf "$HOME_DIR/.cache/pip" 2>/dev/null || true
            rm -rf "$HOME_DIR/.cache/torch" 2>/dev/null || true
            rm -rf "$HOME_DIR/.cache/xdg" 2>/dev/null || true

            find "$HOME_DIR" -maxdepth 1 -type f -name 'qa_answer_*.txt' -delete 2>/dev/null || true

            rm -rf /tmp/"$USERNAME"-venv-* 2>/dev/null || true
            rm -rf /tmp/"$USERNAME"-pip-* 2>/dev/null || true
            rm -rf /tmp/"$USERNAME"-hf-* 2>/dev/null || true
            rm -rf /tmp/"$USERNAME"-job-* 2>/dev/null || true

            mkdir -p "$HOME_DIR/model-store/models"
            mkdir -p "$HOME_DIR/model-store/hf-home"
            mkdir -p "$HOME_DIR/model-store/xdg"

            echo "[CLEANUP] done"
            echo "[CLEANUP] HOME model-store:"
            du -sh "$HOME_DIR/model-store" 2>/dev/null || true
            echo "[CLEANUP] filesystem usage:"
            df -h "$HOME_DIR" /tmp 2>/dev/null || true
        """)

    def make_workdir(self) -> str:
        return self.run(r'''
            mkdir -p "$HOME/.qa_launcher"
            mktemp -d -p "$HOME/.qa_launcher" qa_run.XXXXXX
        ''').strip()

    def upload_text(self, remote_dir: str, filename: str, content: str) -> str:
        remote_path = f"{remote_dir}/{filename}"
        self.run(
            f"""
            set -euo pipefail
            cat > {shlex.quote(remote_path)}
            chmod 600 {shlex.quote(remote_path)}
            """,
            input_text=content,
        )
        return remote_path

    def mkdir_p(self, remote_path: str) -> None:
        self.run(f"mkdir -p {shlex.quote(remote_path)}")

    def upload_file(self, local_path: Path, remote_path: str) -> str:
        local_path = Path(local_path)
        if not local_path.exists():
            raise FileNotFoundError(f"Missing local file: {local_path.resolve()}")

        remote_parent = str(Path(remote_path).parent)
        self.mkdir_p(remote_parent)

        p = subprocess.run(
            ["scp", str(local_path), f"{self.ssh_host}:{remote_path}"],
            text=True,
            capture_output=True,
        )
        if p.returncode != 0:
            raise RuntimeError(
                f"Failed to upload file.\n"
                f"local_path={local_path.resolve()}\n"
                f"remote_path={remote_path}\n"
                f"{p.stderr}"
            )

        self.run(f"chmod 600 {shlex.quote(remote_path)}")
        return remote_path

    def write_remote_script(self, remote_dir: str, script_name: str, script_text: str) -> str:
        remote_path = f"{remote_dir}/{script_name}"
        self.run(
            f"cat > {shlex.quote(remote_path)} <<'EOF'\n{script_text}\nEOF\n"
            f"chmod 700 {shlex.quote(remote_path)}"
        )
        return remote_path

    def submit_job(
        self,
        remote_dir: str,
        script_path: str,
        gpu_request: int,
        walltime: str,
        host_predicate: str,
    ) -> tuple[str, str]:
        out = self.run(f"""
            oarsub -l "host=1/gpu={gpu_request},walltime={walltime}" -p {shlex.quote(host_predicate)} \
              -d {shlex.quote(remote_dir)} \
              -O {shlex.quote(remote_dir)}/stdout.%jobid%.txt \
              -E {shlex.quote(remote_dir)}/stderr.%jobid%.txt \
              {shlex.quote(script_path)}
        """)
        m = re.search(r"OAR_JOB_ID=(\d+)", out)
        if not m:
            raise RuntimeError(f"Could not parse OAR_JOB_ID.\n--- oarsub output ---\n{out}")
        return m.group(1), out

    def oar_field(self, jobid: str, field: str) -> str:
        field = re.sub(r"[^A-Za-z0-9_]", "", field)
        return self.run_ok(f"""
            oarstat -j {shlex.quote(str(jobid))} -f 2>/dev/null |
            sed -n 's/^[[:space:]]*{field}[[:space:]]*=[[:space:]]*//p' |
            head -n 1 || true
        """).strip()

    def job_state(self, jobid: str) -> str:
        return self.oar_field(jobid, "state") or "UNKNOWN"

    def job_host(self, jobid: str) -> str:
        return self.oar_field(jobid, "assigned_hostnames")

    def wait_for_answer(
        self,
        jobid: str,
        answer_path: str,
        status_path: str,
        outputs_dir: str,
        remote_dir: str,
        poll_seconds: int,
        max_wait_minutes: int,
    ) -> tuple[str, str]:
        deadline = time.time() + max_wait_minutes * 60
        final_state = "UNKNOWN"
        seen_finishing = False
        terminated_seen_at = None
        live_log_path = f"{remote_dir}/live.log"
        last_live_len = 0
        last_live_text = ""

        def is_final_status(text: str) -> bool:
            t = (text or "").strip()
            return t.startswith("OK") or t.startswith("PARTIAL_OK") or t.startswith("ERROR")

        while time.time() < deadline:
            state = self.job_state(jobid)
            host = self.job_host(jobid)
            ts = datetime.datetime.now().strftime("%H:%M:%S")
            print(f"[{ts}] state={state}" + (f" host={host}" if host else ""))
            final_state = state

            if state == "Finishing":
                seen_finishing = True

            live_text = self.read_remote_text(live_log_path)
            if len(live_text) < last_live_len:
                last_live_len = 0
            if len(live_text) > last_live_len:
                new_live = live_text[last_live_len:]
                if new_live.strip():
                    print("\n[live.log]")
                    print(new_live, end="" if new_live.endswith("\n") else "\n", flush=True)
                last_live_len = len(live_text)
                last_live_text = live_text

            answer_text = self.read_remote_text(answer_path)
            status_text = self.read_remote_text(status_path)

            if answer_text.strip():
                return state, answer_text

            if is_final_status(status_text):
                return state, status_text

            if state == "Error":
                stderr = self.read_remote_text(f"{remote_dir}/stderr.{jobid}.txt")
                stdout = self.read_remote_text(f"{remote_dir}/stdout.{jobid}.txt")
                raise RuntimeError(
                    "Remote job entered Error state.\n\n"
                    f"--- STDOUT ---\n{stdout[-8000:]}\n\n"
                    f"--- STDERR ---\n{stderr[-8000:]}"
                )

            if state == "Terminated":
                if terminated_seen_at is None:
                    terminated_seen_at = time.time()

                outputs_ready = self.remote_exists(outputs_dir)
                status_ready = self.remote_exists(status_path)
                answer_ready = self.remote_exists(answer_path)

                if answer_ready or status_ready or outputs_ready:
                    answer_text = self.read_remote_text(answer_path)
                    status_text = self.read_remote_text(status_path)
                    if answer_text.strip():
                        return state, answer_text
                    if status_text.strip():
                        return state, status_text
                    if last_live_text.strip():
                        return state, last_live_text

                if seen_finishing and (time.time() - terminated_seen_at) > 10:
                    stderr = self.read_remote_text(f"{remote_dir}/stderr.{jobid}.txt")
                    stdout = self.read_remote_text(f"{remote_dir}/stdout.{jobid}.txt")
                    raise RuntimeError(
                        "Remote job terminated after finishing state but produced no outputs.\n\n"
                        f"--- STDOUT ---\n{stdout[-8000:]}\n\n"
                        f"--- STDERR ---\n{stderr[-8000:]}"
                    )

                if (time.time() - terminated_seen_at) > 20:
                    stderr = self.read_remote_text(f"{remote_dir}/stderr.{jobid}.txt")
                    stdout = self.read_remote_text(f"{remote_dir}/stdout.{jobid}.txt")
                    raise RuntimeError(
                        "Remote job terminated before producing outputs.\n\n"
                        f"--- STDOUT ---\n{stdout[-8000:]}\n\n"
                        f"--- STDERR ---\n{stderr[-8000:]}"
                    )

            time.sleep(poll_seconds)

        answer_text = self.read_remote_text(answer_path)
        if answer_text.strip():
            return final_state, answer_text

        status_text = self.read_remote_text(status_path)
        if status_text.strip():
            return final_state, status_text

        return final_state, last_live_text

    def copy_remote_dir_to_local(self, remote_dir: str, local_dir: Path) -> Path:
        local_dir.mkdir(parents=True, exist_ok=True)
        rsync_cmd = [
            "rsync", "-az", "-e", "ssh",
            f"{self.ssh_host}:{remote_dir}/",
            str(local_dir) + "/",
        ]
        p = subprocess.run(rsync_cmd, text=True, capture_output=True)
        if p.returncode != 0:
            raise RuntimeError(
                f"Failed to copy remote directory.\n"
                f"remote_dir={remote_dir}\n"
                f"local_dir={local_dir}\n"
                f"{p.stderr}"
            )
        return local_dir


# =========================================================
# DISCOVER ONE PAPER SOURCE PER paper_xxx FOLDER
# =========================================================
REVIEW_SKIP_TOKENS = (
    "review",
    "reviewer",
    "separated_reviews",
    ".error",
    "answer",
    "status",
    "summary",
)
VALID_SUFFIXES = {".pdf", ".txt", ".md", ".json"}


def pretty_path(path: Path) -> str:
    path = Path(path).resolve()
    try:
        return path.relative_to(Path.cwd().resolve()).as_posix()
    except Exception:
        return path.as_posix()


def is_review_like_file(path: Path) -> bool:
    name = path.name.lower()
    return any(tok in name for tok in REVIEW_SKIP_TOKENS)


def is_paper_candidate(path: Path) -> bool:
    if not path.is_file():
        return False
    if path.suffix.lower() not in VALID_SUFFIXES:
        return False
    if is_review_like_file(path):
        return False
    return True


def candidate_score(path: Path) -> tuple:
    name = path.name.lower()
    suffix = path.suffix.lower()
    score = 0

    if "article_extracted_reader_text_with_code" in name:
        score += 300
    elif "article_extracted_pure_text" in name:
        score += 260
    elif "article_extracted_full" in name:
        score += 230
    elif "article_extracted" in name:
        score += 200
    elif "article" in name:
        score += 40
    elif "paper" in name or "submission" in name or "manuscript" in name:
        score += 30

    if "bundle" in name:
        score += 100
    if "clean" in name:
        score += 20

    if suffix == ".txt":
        score += 20
    elif suffix == ".md":
        score += 12
    elif suffix == ".pdf":
        score += 8
    elif suffix == ".json":
        score += 5

    return (score, -len(name))


def extract_paper_id(path: Path) -> str:
    candidates = [path.parent.name, path.parent.parent.name, path.stem, path.name]
    for item in candidates:
        m = re.search(r"(\d+)", str(item))
        if m:
            return str(int(m.group(1)))
    raise ValueError(f"Could not infer paper id from: {path}")


def discover_paper_sources_from_root(root: Path) -> list[dict]:
    root = Path(root)
    if not root.exists():
        print(f"[WARN] missing paper root: {pretty_path(root)}")
        return []

    paper_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]
    if not paper_dirs:
        print(f"[WARN] no paper subfolders found under: {pretty_path(root)}")
        return []

    records = []
    for paper_dir in paper_dirs:
        candidates = [p.resolve() for p in paper_dir.rglob("*") if is_paper_candidate(p)]
        if not candidates:
            print(f"[WARN] no paper source found in {pretty_path(paper_dir)}")
            continue

        candidates = sorted(candidates, key=candidate_score, reverse=True)
        chosen = candidates[0]

        records.append({
            "paper_root": root.resolve(),
            "paper_id": extract_paper_id(chosen),
            "paper_dir": paper_dir.resolve(),
            "paper_path": chosen,
            "all_candidates": candidates,
        })

    return records


def dedupe_records(records: list[dict]) -> list[dict]:
    by_paper_id = {}
    for rec in records:
        paper_id = str(rec["paper_id"])
        current = by_paper_id.get(paper_id)

        if current is None:
            by_paper_id[paper_id] = rec
            continue

        current_score = candidate_score(current["paper_path"])
        new_score = candidate_score(rec["paper_path"])

        if new_score > current_score:
            by_paper_id[paper_id] = rec

    return [by_paper_id[k] for k in sorted(by_paper_id, key=lambda x: int(x))]


def discover_paper_sources(roots: list[Path] | tuple[Path, ...]) -> list[dict]:
    all_records = []
    for root in roots:
        all_records.extend(discover_paper_sources_from_root(Path(root)))
    return dedupe_records(all_records)


def ensure_local_dirs() -> None:
    JDH_ROOT.mkdir(parents=True, exist_ok=True)
    PRIMARY_PAPER_FILES_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_COLLECTIONS_DIR.mkdir(parents=True, exist_ok=True)
    if QDRANT_MODE == "local":
        LOCAL_QDRANT_DIR.mkdir(parents=True, exist_ok=True)


# =========================================================
# MAIN
# =========================================================
def main() -> None:
    ensure_local_dirs()

    paper_records = discover_paper_sources(PAPER_FILES_ROOT_CANDIDATES)
    print("Found paper records:", len(paper_records))
    for rec in paper_records:
        print(f"\npaper_id={rec['paper_id']} source={pretty_path(rec['paper_path'])}")
        for c in rec["all_candidates"][:5]:
            print("   -", c.name)

    if not paper_records:
        raise RuntimeError("No paper sources found.")

    run_sh = r"""#!/usr/bin/env bash
set -euo pipefail

WORKDIR="$PWD"
ANSWER_FILE="$WORKDIR/answer.txt"
STATUS_FILE="$WORKDIR/status.txt"
SUMMARY_FILE="$WORKDIR/summary.json"
LIVE_LOG="$WORKDIR/live.log"
: > "$LIVE_LOG"

exec > >(tee -a "$LIVE_LOG") 2>&1

echo "BOOTSTRAP_STARTED" > "$STATUS_FILE"

export MODEL_ID="$(cat "$WORKDIR/model.txt")"
export EMBEDDING_MODEL_ID="$(cat "$WORKDIR/embedding_model.txt")"
export EMBEDDING_DIM="$(cat "$WORKDIR/embedding_dim.txt")"
export QDRANT_MODE="$(cat "$WORKDIR/qdrant_mode.txt")"
export QDRANT_URL="$(cat "$WORKDIR/qdrant_url.txt")"
export QDRANT_API_KEY="$(cat "$WORKDIR/qdrant_api_key.txt")"

export TORCH_INDEX_URL="$(cat "$WORKDIR/torch_index_url.txt")"
export TORCH_VERSION="$(cat "$WORKDIR/torch_version.txt")"
export TORCHVISION_VERSION="$(cat "$WORKDIR/torchvision_version.txt")"
export TORCHAUDIO_VERSION="$(cat "$WORKDIR/torchaudio_version.txt")"
export HSA_OVERRIDE_GFX_VERSION_VALUE="$(cat "$WORKDIR/hsa_override_gfx_version.txt")"

MODEL_SLUG="$(printf '%s' "$MODEL_ID" | sed 's#[/:]#_#g' | tr -cd '[:alnum:]_.-')"
EMBED_SLUG="$(printf '%s' "$EMBEDDING_MODEL_ID" | sed 's#[/:]#_#g' | tr -cd '[:alnum:]_.-')"

export MODEL_STORE_ROOT="$HOME/model-store"
export HF_HOME="$MODEL_STORE_ROOT/hf-home"
export HF_HUB_CACHE="$HF_HOME/hub"
export HF_DATASETS_CACHE="$HF_HOME/datasets"
export XDG_CACHE_HOME="$MODEL_STORE_ROOT/xdg"
export PIP_CACHE_DIR="/tmp/$USER-pip-$OAR_JOB_ID"

export CHUNK_MODEL_DIR="$MODEL_STORE_ROOT/models/$MODEL_SLUG"
export EMBEDDING_MODEL_DIR="$MODEL_STORE_ROOT/models/$EMBED_SLUG"

mkdir -p \
  "$HF_HUB_CACHE" \
  "$HF_DATASETS_CACHE" \
  "$XDG_CACHE_HOME" \
  "$PIP_CACHE_DIR" \
  "$CHUNK_MODEL_DIR" \
  "$EMBEDDING_MODEL_DIR"

export HIP_VISIBLE_DEVICES=0,1
export ROCR_VISIBLE_DEVICES=0,1
export CUDA_VISIBLE_DEVICES=0,1
export HSA_OVERRIDE_GFX_VERSION="$HSA_OVERRIDE_GFX_VERSION_VALUE"
export AMD_SERIALIZE_KERNEL=3
export TORCH_SHOW_CPP_STACKTRACES=1
export PYTORCH_HIP_ALLOC_CONF="expandable_segments:True,max_split_size_mb:128"
export TOKENIZERS_PARALLELISM=false
export PYTHONUNBUFFERED=1
export PYTHONFAULTHANDLER=1

echo "HOSTNAME=$(hostname)"
echo "WORKDIR=$WORKDIR"
echo "MODEL_ID=$MODEL_ID"
echo "EMBEDDING_MODEL_ID=$EMBEDDING_MODEL_ID"
echo "MODEL_STORE_ROOT=$MODEL_STORE_ROOT"
echo "HF_HOME=$HF_HOME"
echo "CHUNK_MODEL_DIR=$CHUNK_MODEL_DIR"
echo "EMBEDDING_MODEL_DIR=$EMBEDDING_MODEL_DIR"
echo "PIP_CACHE_DIR=$PIP_CACHE_DIR"
echo "[DISK USAGE BEFORE DOWNLOAD]"
du -sh "$MODEL_STORE_ROOT" 2>/dev/null || true
df -h "$HOME" /tmp 2>/dev/null || true
echo "[OAR ENV]"
env | sort | grep '^OAR_' || true
echo "[DEVICE NODES]"
ls -l /dev/kfd /dev/dri/render* 2>/dev/null || true

VENV="/tmp/$USER-venv-$OAR_JOB_ID"

cleanup_tmp() {
  RC=$?
  echo "[CLEANUP] removing current job tmp files"
  rm -rf "$VENV" 2>/dev/null || true
  rm -rf "$PIP_CACHE_DIR" 2>/dev/null || true
  exit $RC
}
trap cleanup_tmp EXIT

python3 -m venv "$VENV"
source "$VENV/bin/activate"

python -m pip -q install -U pip wheel setuptools
python -m pip -q install --no-cache-dir \
  "torch==${TORCH_VERSION}" \
  "torchvision==${TORCHVISION_VERSION}" \
  "torchaudio==${TORCHAUDIO_VERSION}" \
  --index-url "${TORCH_INDEX_URL}"

python -m pip -q install --no-cache-dir \
  "transformers>=4.52" \
  "accelerate>=1.7" \
  "safetensors>=0.4.3" \
  "sentencepiece" \
  "huggingface_hub>=0.23" \
  "sentence-transformers>=3.0" \
  "qdrant-client>=1.9.1" \
  "pypdf" \
  "numpy"

python - <<'PY'
import os
from huggingface_hub import snapshot_download

chunk_dir = snapshot_download(
    repo_id=os.environ["MODEL_ID"],
    local_dir=os.environ["CHUNK_MODEL_DIR"],
)
embed_dir = snapshot_download(
    repo_id=os.environ["EMBEDDING_MODEL_ID"],
    local_dir=os.environ["EMBEDDING_MODEL_DIR"],
)

print("CHUNK MODEL downloaded to:", chunk_dir)
print("EMBED MODEL downloaded to:", embed_dir)
PY

echo "[DISK USAGE AFTER DOWNLOAD]"
du -sh "$CHUNK_MODEL_DIR" 2>/dev/null || true
du -sh "$EMBEDDING_MODEL_DIR" 2>/dev/null || true
du -sh "$MODEL_STORE_ROOT" 2>/dev/null || true
df -h "$HOME" /tmp 2>/dev/null || true

python - <<'PY'
from __future__ import annotations

import json
import os
import re
import traceback
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence

import numpy as np
import torch
from pypdf import PdfReader
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, PointStruct, VectorParams
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer


workdir = Path.cwd()
manifest = json.loads((workdir / "manifest.json").read_text(encoding="utf-8"))
outputs_dir = workdir / "outputs"
paper_collections_dir = outputs_dir / "paper_collections"
paper_qdrant_dir = outputs_dir / "paper_qdrant_db"
paper_collections_dir.mkdir(parents=True, exist_ok=True)
paper_qdrant_dir.mkdir(parents=True, exist_ok=True)

answer_file = workdir / "answer.txt"
status_file = workdir / "status.txt"
summary_file = workdir / "summary.json"

embedding_dim = int((workdir / "embedding_dim.txt").read_text().strip())
qdrant_mode = (workdir / "qdrant_mode.txt").read_text().strip()
qdrant_url = (workdir / "qdrant_url.txt").read_text().strip()
qdrant_api_key = (workdir / "qdrant_api_key.txt").read_text().strip()

chunk_model_path = os.environ.get("CHUNK_MODEL_DIR", "").strip()
embedding_model_path = os.environ.get("EMBEDDING_MODEL_DIR", "").strip()

SECTION_PATTERNS = [
    r"abstract", r"introduction", r"background", r"related work", r"literature review",
    r"method", r"methods", r"methodology", r"approach", r"experimental setup",
    r"experiments", r"evaluation", r"results", r"discussion", r"limitations",
    r"conclusion", r"conclusions", r"future work", r"appendix", r"references",
]
SKIP_SECTION_TITLES = {"references"}

results = []
errors = []


@dataclass
class PaperChunk:
    chunk_id: str
    paper_id: str
    source_file: str
    section_index: int
    section_title: str
    section_path: str
    chunk_index: int
    local_chunk_index: int
    chunk_label: str
    chunk_summary: str
    chunk_type: str
    word_count: int
    text: str


def atomic_write(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(content, encoding="utf-8")
    tmp.replace(path)


def atomic_json(path: Path, obj: Any) -> None:
    atomic_write(path, json.dumps(obj, indent=2, ensure_ascii=False) + "\n")


def finalize(current_status: str = "OK") -> None:
    summary = {
        "processed_count": len(results),
        "error_count": len(errors),
        "results": results,
        "errors": errors,
    }
    atomic_json(summary_file, summary)
    atomic_json(answer_file, summary)
    atomic_write(status_file, current_status + "\n")
    try:
        home_answer = Path.home() / f"qa_answer_{os.environ['OAR_JOB_ID']}.txt"
        home_answer.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    except Exception:
        pass


def fail_fast(msg: str) -> None:
    errors.append({
        "source_file": "__startup__",
        "error": msg,
        "traceback": msg,
    })
    finalize("ERROR")
    raise RuntimeError(msg)


def clean_text(text: Any) -> str:
    if text is None:
        return ""
    text = str(text).replace("\r\n", "\n").replace("\r", "\n").replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def word_count(text: str) -> int:
    return len(re.findall(r"\S+", text))


def parse_paper_id(path: Path) -> str:
    m = re.search(r"(\d+)", path.stem)
    return m.group(1) if m else path.stem


def safe_collection_name(name: str) -> str:
    name = re.sub(r"[^a-zA-Z0-9_-]+", "_", name)
    return name[:200].strip("_") or "paper_collection"


def normalize_heading(line: str) -> str:
    line = clean_text(line)
    line = re.sub(r"^\d+(?:\.\d+)*\s+", "", line)
    return line.strip()


def looks_like_heading(line: str) -> bool:
    line = normalize_heading(line)
    if not line or len(line.split()) > 8:
        return False
    low = line.lower().strip(":")
    for pattern in SECTION_PATTERNS:
        if re.fullmatch(pattern, low):
            return True
    return line.isupper() and len(line.split()) <= 6


def split_sentences(text: str) -> List[str]:
    text = clean_text(text)
    if not text:
        return []
    pattern = re.compile(r"(?<=[.!?])\s+(?=(?:[A-Z0-9\[(\"]|Figure|Fig\.|Table|Eq\.|Equation))")
    return [clean_text(p) for p in pattern.split(text) if clean_text(p)]


def detect_chunk_type(section_title: str, text: str) -> str:
    title = section_title.lower()
    stripped = text.strip().lower()
    if "abstract" in title:
        return "abstract"
    if "conclusion" in title:
        return "conclusion"
    if "limitation" in title:
        return "limitations"
    if "appendix" in title:
        return "appendix"
    if "reference" in title:
        return "references"
    if re.match(r"^(figure|fig\.)\s*\d+", stripped):
        return "figure_caption"
    if re.match(r"^table\s*\d+", stripped):
        return "table_caption"
    return "body"


def read_paper_text(path: Path) -> str:
    suffix = path.suffix.lower()

    if suffix in {".txt", ".md"}:
        return clean_text(path.read_text(encoding="utf-8", errors="ignore"))

    if suffix == ".json":
        data = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
        if isinstance(data, dict):
            for key in ["paper_text", "text", "full_text", "content", "body"]:
                if key in data and data[key]:
                    return clean_text(data[key])
        raise ValueError(f"Could not find paper text field in JSON file: {path}")

    if suffix == ".pdf":
        reader = PdfReader(str(path))
        return clean_text("\n\n".join(page.extract_text() or "" for page in reader.pages))

    raise ValueError(f"Unsupported file type: {path}")


def split_into_sections(paper_text: str) -> List[Dict[str, str]]:
    lines = [line.rstrip() for line in clean_text(paper_text).split("\n")]
    sections: List[Dict[str, str]] = []
    current_title = "Front Matter"
    current_lines: List[str] = []

    for line in lines:
        stripped = line.strip()
        if looks_like_heading(stripped):
            content = clean_text("\n".join(current_lines))
            if content:
                sections.append({"section_title": current_title, "content": content})
            current_title = normalize_heading(stripped)
            current_lines = []
        else:
            current_lines.append(line)

    content = clean_text("\n".join(current_lines))
    if content:
        sections.append({"section_title": current_title, "content": content})

    if not sections and clean_text(paper_text):
        sections = [{"section_title": "Full Paper", "content": clean_text(paper_text)}]

    return sections


def paragraph_units(section_text: str, max_unit_words: int = 1300) -> List[str]:
    paragraphs = [clean_text(p) for p in re.split(r"\n\s*\n+", clean_text(section_text)) if clean_text(p)]
    if not paragraphs:
        return [clean_text(section_text)] if clean_text(section_text) else []

    units, current, current_words = [], [], 0
    for paragraph in paragraphs:
        pw = word_count(paragraph)
        if current and current_words + pw > max_unit_words:
            units.append("\n\n".join(current).strip())
            current, current_words = [paragraph], pw
        else:
            current.append(paragraph)
            current_words += pw

    if current:
        units.append("\n\n".join(current).strip())

    return units


class QwenSectionChunker:
    def __init__(self, model_path: str, max_new_tokens: int = 1200, prompt_sentence_limit: int = 120) -> None:
        self.model_path = model_path
        self.max_new_tokens = max_new_tokens
        self.prompt_sentence_limit = prompt_sentence_limit

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_path,
            trust_remote_code=True,
            use_fast=False,
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_path,
            trust_remote_code=True,
            device_map="auto",
            max_memory={0: "58GiB", 1: "58GiB", "cpu": "64GiB"},
            use_safetensors=True,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        )
        self.model.eval()

    def _prompt(self, section_title: str, numbered_sentences: str) -> str:
        return (
            "You are segmenting one academic paper section for retrieval.\n"
            "You receive numbered sentences from exactly one section.\n"
            "Group contiguous sentence ids into semantically coherent chunks.\n\n"
            "Rules:\n"
            "- Use every sentence exactly once.\n"
            "- Keep the original order.\n"
            "- Chunks must be contiguous ranges.\n"
            "- Do not overlap ranges.\n"
            "- Prefer chunks around 120 to 250 words when possible.\n"
            "- Keep claims, evidence, equations, figure mentions, and their explanation together.\n"
            "- Keep background separate from method, results, discussion, and limitations when possible.\n"
            "- Output JSON only.\n\n"
            "Return this exact schema:\n"
            "{\n"
            '  "chunks": [\n'
            "    {\n"
            '      "start": 1,\n'
            '      "end": 4,\n'
            '      "label": "short label",\n'
            '      "summary": "one-sentence summary"\n'
            "    }\n"
            "  ]\n"
            "}\n\n"
            f"Section title: {section_title}\n\n"
            "Sentences:\n"
            f"{numbered_sentences}\n"
        )

    def _extract_json(self, text: str) -> Dict[str, Any]:
        text = text.strip()
        if text.startswith("```"):
            text = re.sub(r"^```(?:json)?", "", text).strip()
            text = re.sub(r"```$", "", text).strip()
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise ValueError("No JSON object found.")
        return json.loads(match.group(0))

    def _fallback_ranges(self, sentences: Sequence[str], min_words: int = 120, max_words: int = 260) -> List[Dict[str, Any]]:
        chunks = []
        start, current_words = 1, 0

        for idx, sentence in enumerate(sentences, start=1):
            sw = word_count(sentence)
            if idx > start and current_words + sw > max_words:
                chunks.append({
                    "start": start,
                    "end": idx - 1,
                    "label": "sequential span",
                    "summary": "Fallback contiguous span.",
                })
                start, current_words = idx, sw
            else:
                current_words += sw

        if sentences:
            chunks.append({
                "start": start,
                "end": len(sentences),
                "label": "sequential span",
                "summary": "Fallback contiguous span.",
            })

        if len(chunks) >= 2:
            last_words = sum(word_count(sentences[i - 1]) for i in range(chunks[-1]["start"], chunks[-1]["end"] + 1))
            if last_words < min_words:
                chunks[-2]["end"] = chunks[-1]["end"]
                chunks.pop()

        return chunks

    def _validate_ranges(self, raw_chunks: Any, n_sentences: int) -> List[Dict[str, Any]] | None:
        if not isinstance(raw_chunks, list) or not raw_chunks:
            return None

        normalized = []
        expected = 1

        for item in raw_chunks:
            try:
                start, end = int(item["start"]), int(item["end"])
            except Exception:
                return None

            if start != expected or end < start or end > n_sentences:
                return None

            normalized.append({
                "start": start,
                "end": end,
                "label": clean_text(item.get("label") or "semantic span") or "semantic span",
                "summary": clean_text(item.get("summary") or "") or "",
            })
            expected = end + 1

        return normalized if expected == n_sentences + 1 else None

    def propose_ranges(self, section_title: str, sentences: Sequence[str]) -> List[Dict[str, Any]]:
        if not sentences:
            return []

        if len(sentences) <= 5:
            return [{
                "start": 1,
                "end": len(sentences),
                "label": "short section",
                "summary": "Short section kept as one chunk.",
            }]

        numbered = "\n".join(f"[{i}] {s}" for i, s in enumerate(sentences, start=1))
        messages = [
            {"role": "system", "content": "Return strict JSON only."},
            {"role": "user", "content": self._prompt(section_title, numbered)},
        ]

        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self.tokenizer(prompt, return_tensors="pt")

        first_device = next(iter(self.model.hf_device_map.values()))
        if isinstance(first_device, str):
            model_inputs = {k: v.to(first_device) for k, v in model_inputs.items()}
        else:
            model_inputs = {k: v.to(f"cuda:{first_device}") for k, v in model_inputs.items()}

        with torch.no_grad():
            output_ids = self.model.generate(
                **model_inputs,
                do_sample=False,
                temperature=None,
                max_new_tokens=self.max_new_tokens,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        generated_ids = output_ids[0][model_inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        try:
            data = self._extract_json(text)
            valid = self._validate_ranges(data.get("chunks"), len(sentences))
            if valid is not None:
                return valid
        except Exception:
            pass

        return self._fallback_ranges(sentences)

    def chunk_section(self, section_title: str, section_text: str) -> List[Dict[str, Any]]:
        section_text = clean_text(section_text)
        if not section_text:
            return []

        units = paragraph_units(section_text)
        all_chunks = []

        for unit in units:
            sentences = split_sentences(unit)
            if not sentences:
                continue

            if len(sentences) > self.prompt_sentence_limit:
                ranges = []
                start_idx = 0
                while start_idx < len(sentences):
                    sub = sentences[start_idx:start_idx + self.prompt_sentence_limit]
                    sub_ranges = self.propose_ranges(section_title, sub)
                    for item in sub_ranges:
                        ranges.append({
                            "start": item["start"] + start_idx,
                            "end": item["end"] + start_idx,
                            "label": item["label"],
                            "summary": item["summary"],
                        })
                    start_idx += self.prompt_sentence_limit
            else:
                ranges = self.propose_ranges(section_title, sentences)

            for item in ranges:
                local_sentences = sentences[item["start"] - 1:item["end"]]
                chunk_text = " ".join(local_sentences).strip()
                all_chunks.append({
                    "label": item["label"],
                    "summary": item["summary"],
                    "text": chunk_text,
                    "word_count": word_count(chunk_text),
                })

        return all_chunks


class LocalEmbedder:
    def __init__(self, model_path: str, batch_size: int = 16) -> None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_path, device=device)
        self.batch_size = batch_size

    def encode(self, texts: Sequence[str]) -> np.ndarray:
        return np.asarray(
            self.model.encode(
                list(texts),
                batch_size=self.batch_size,
                show_progress_bar=False,
                normalize_embeddings=True,
                convert_to_numpy=True,
            )
        )


def make_qdrant_client() -> QdrantClient:
    if qdrant_mode == "server":
        if not qdrant_url:
            raise ValueError("QDRANT_MODE is 'server' but QDRANT_URL is empty.")
        kwargs = {"url": qdrant_url}
        if qdrant_api_key:
            kwargs["api_key"] = qdrant_api_key
        return QdrantClient(**kwargs)
    return QdrantClient(path=str(paper_qdrant_dir))


def ensure_collection(client: QdrantClient, collection_name: str):
    existing = [c.name for c in client.get_collections().collections]
    if collection_name in existing:
        client.delete_collection(collection_name=collection_name)

    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )


try:
    if not chunk_model_path:
        fail_fast("Missing CHUNK_MODEL_DIR in environment.")
    if not embedding_model_path:
        fail_fast("Missing EMBEDDING_MODEL_DIR in environment.")

    chunker = QwenSectionChunker(chunk_model_path)
    embedder = LocalEmbedder(embedding_model_path)
    qdrant_client = make_qdrant_client()

    global_chunk_index = 1

    for entry in manifest:
        try:
            source_file = entry["source_file"]
            paper_path = Path(entry["remote_file"])
            paper_id = str(entry.get("paper_id") or parse_paper_id(paper_path))

            paper_text = read_paper_text(paper_path)
            sections = split_into_sections(paper_text)
            paper_chunks: List[PaperChunk] = []

            for section_index, section in enumerate(sections, start=1):
                section_title = clean_text(section["section_title"]) or f"Section {section_index}"
                if section_title.lower() in SKIP_SECTION_TITLES:
                    continue

                semantic_chunks = chunker.chunk_section(section_title, section["content"])

                for local_idx, chunk in enumerate(semantic_chunks, start=1):
                    text = (
                        f"Paper ID: {paper_id}\n"
                        f"Section: {section_title}\n"
                        f"Chunk Label: {chunk['label']}\n\n"
                        f"{chunk['text']}"
                    ).strip()

                    paper_chunks.append(
                        PaperChunk(
                            chunk_id=f"paper_{paper_id}__chunk_{global_chunk_index}",
                            paper_id=paper_id,
                            source_file=source_file,
                            section_index=section_index,
                            section_title=section_title,
                            section_path=section_title,
                            chunk_index=global_chunk_index,
                            local_chunk_index=local_idx,
                            chunk_label=chunk["label"],
                            chunk_summary=chunk["summary"],
                            chunk_type=detect_chunk_type(section_title, chunk["text"]),
                            word_count=chunk["word_count"],
                            text=text,
                        )
                    )
                    global_chunk_index += 1

            paper_dir = paper_collections_dir / f"paper_{paper_id}"
            paper_dir.mkdir(parents=True, exist_ok=True)

            with (paper_dir / "chunks.jsonl").open("w", encoding="utf-8") as f:
                for chunk in paper_chunks:
                    f.write(json.dumps(asdict(chunk), ensure_ascii=False) + "\n")

            manifest_obj = {
                "paper_id": paper_id,
                "source_file": source_file,
                "num_chunks": len(paper_chunks),
                "sections": sorted({c.section_title for c in paper_chunks}),
            }
            atomic_json(paper_dir / "manifest.json", manifest_obj)

            collection_name = safe_collection_name(f"paper_{paper_id}")
            ensure_collection(qdrant_client, collection_name)

            docs = [c.text for c in paper_chunks]
            if docs:
                embs = embedder.encode(docs)
                if embs.shape[1] != embedding_dim:
                    raise ValueError(
                        f"Embedding dimension mismatch: expected {embedding_dim}, got {embs.shape[1]}"
                    )

                points = []
                for i, chunk in enumerate(paper_chunks, start=1):
                    payload = {
                        "chunk_id": chunk.chunk_id,
                        "paper_id": chunk.paper_id,
                        "source_file": chunk.source_file,
                        "section_index": chunk.section_index,
                        "section_title": chunk.section_title,
                        "section_path": chunk.section_path,
                        "chunk_index": chunk.chunk_index,
                        "local_chunk_index": chunk.local_chunk_index,
                        "chunk_label": chunk.chunk_label,
                        "chunk_summary": chunk.chunk_summary,
                        "chunk_type": chunk.chunk_type,
                        "word_count": chunk.word_count,
                        "text": chunk.text,
                    }

                    points.append(
                        PointStruct(
                            id=i,
                            vector=embs[i - 1].tolist(),
                            payload=payload,
                        )
                    )

                batch_size = 64
                for start in range(0, len(points), batch_size):
                    end = start + batch_size
                    qdrant_client.upsert(
                        collection_name=collection_name,
                        points=points[start:end],
                        wait=True,
                    )

            results.append({
                "source_file": source_file,
                "paper_id": paper_id,
                "chunk_count": len(paper_chunks),
                "collection_name": collection_name,
                "chunks_jsonl": str((paper_dir / "chunks.jsonl").relative_to(workdir)),
                "manifest_json": str((paper_dir / "manifest.json").relative_to(workdir)),
            })
            print(f"PROCESSED {source_file} -> {len(paper_chunks)} chunks", flush=True)

        except Exception as e:
            errors.append({
                "source_file": entry.get("source_file", "__unknown__"),
                "error": str(e),
                "traceback": traceback.format_exc(),
            })
            print(f"FAILED {entry.get('source_file', '__unknown__')}", flush=True)
            print(traceback.format_exc(), flush=True)

    finalize("PARTIAL_OK" if errors else "OK")

except Exception as e:
    errors.append({
        "source_file": "__startup__",
        "error": str(e),
        "traceback": traceback.format_exc(),
    })
    finalize("ERROR")
    print("FAILED DURING STARTUP", flush=True)
    print(traceback.format_exc(), flush=True)

finally:
    try:
        import gc

        if "chunker" in locals():
            if hasattr(chunker, "model"):
                del chunker.model
            if hasattr(chunker, "tokenizer"):
                del chunker.tokenizer

        if "embedder" in locals():
            if hasattr(embedder, "model"):
                del embedder.model

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as cleanup_err:
        print(f"CLEANUP WARNING: {cleanup_err}", flush=True)
PY
"""

    g5k = Grid5000Client()

    print("\nPreparing Grid'5000 HOME...")
    print(g5k.prepare_home())

    remote_workdir = g5k.make_workdir()
    remote_inputs_dir = f"{remote_workdir}/input_papers"
    g5k.mkdir_p(remote_inputs_dir)

    jdh_root_resolved = JDH_ROOT.resolve()

    manifest = []
    for rec in paper_records:
        local_path = rec["paper_path"]
        rel = local_path.relative_to(jdh_root_resolved).as_posix()
        remote_path = f"{remote_inputs_dir}/{rel}"

        print("uploading:", rel)
        g5k.upload_file(local_path, remote_path)

        manifest.append({
            "paper_id": rec["paper_id"],
            "source_file": rel,
            "remote_file": remote_path,
        })

    g5k.upload_text(remote_workdir, "manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))
    g5k.upload_text(remote_workdir, "model.txt", MODEL_ID)
    g5k.upload_text(remote_workdir, "embedding_model.txt", EMBEDDING_MODEL_ID)
    g5k.upload_text(remote_workdir, "embedding_dim.txt", str(EMBEDDING_DIM))
    g5k.upload_text(remote_workdir, "qdrant_mode.txt", QDRANT_MODE)
    g5k.upload_text(remote_workdir, "qdrant_url.txt", QDRANT_URL)
    g5k.upload_text(remote_workdir, "qdrant_api_key.txt", QDRANT_API_KEY)

    g5k.upload_text(remote_workdir, "torch_index_url.txt", TORCH_INDEX_URL)
    g5k.upload_text(remote_workdir, "torch_version.txt", TORCH_VERSION)
    g5k.upload_text(remote_workdir, "torchvision_version.txt", TORCHVISION_VERSION)
    g5k.upload_text(remote_workdir, "torchaudio_version.txt", TORCHAUDIO_VERSION)
    g5k.upload_text(remote_workdir, "hsa_override_gfx_version.txt", HSA_OVERRIDE_GFX_VERSION)

    remote_script_path = g5k.write_remote_script(remote_workdir, "run.sh", run_sh)

    jobid, submit_out = g5k.submit_job(
        remote_dir=remote_workdir,
        script_path=remote_script_path,
        gpu_request=GPU_REQUEST,
        walltime=WALLTIME,
        host_predicate=HOST_PREDICATE,
    )

    print("\n" + submit_out.strip())
    print("jobid:", jobid)

    remote_answer_path = f"{remote_workdir}/answer.txt"
    remote_status_path = f"{remote_workdir}/status.txt"
    remote_outputs_dir = f"{remote_workdir}/outputs"

    state, answer = g5k.wait_for_answer(
        jobid=jobid,
        answer_path=remote_answer_path,
        status_path=remote_status_path,
        outputs_dir=remote_outputs_dir,
        remote_dir=remote_workdir,
        poll_seconds=POLL_SECONDS,
        max_wait_minutes=MAX_WAIT_MINUTES,
    )

    print("\nFinal state:", state)
    print("\nRemote summary:\n", answer)

    if not g5k.remote_exists(remote_outputs_dir):
        raise RuntimeError(f"Remote outputs directory was not created: {remote_outputs_dir}")

    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    local_job_dir = LOCAL_RESULTS_ROOT / f"job_{jobid}_{ts}"
    g5k.copy_remote_dir_to_local(remote_workdir, local_job_dir)
    print("\nDownloaded job folder:", local_job_dir.resolve())

    outputs_root = local_job_dir / "outputs"
    if not outputs_root.exists():
        raise RuntimeError(f"Missing downloaded outputs directory: {outputs_root}")

    remote_collections = outputs_root / "paper_collections"
    remote_qdrant = outputs_root / "paper_qdrant_db"

    if remote_collections.exists():
        if LOCAL_COLLECTIONS_DIR.exists():
            shutil.rmtree(LOCAL_COLLECTIONS_DIR)
        shutil.copytree(remote_collections, LOCAL_COLLECTIONS_DIR)

    if QDRANT_MODE == "local" and remote_qdrant.exists():
        if LOCAL_QDRANT_DIR.exists():
            shutil.rmtree(LOCAL_QDRANT_DIR)
        shutil.copytree(remote_qdrant, LOCAL_QDRANT_DIR)

    print("\nLocal collections:", LOCAL_COLLECTIONS_DIR.resolve())
    if QDRANT_MODE == "local":
        print("Local qdrant dir:", LOCAL_QDRANT_DIR.resolve())
    else:
        print("Qdrant server mode was used:", QDRANT_URL)

    print("\nManifests:")
    for p in sorted(LOCAL_COLLECTIONS_DIR.rglob("manifest.json")):
        print(" -", p)


if __name__ == "__main__":
    main()

[WARN] missing paper root: paper_reviews
Found paper records: 3

paper_id=1 source=review_outputs/paper_reviews/paper_001/article_extracted_reader_text_with_code.txt
   - article_extracted_reader_text_with_code.txt
   - article_extracted_reader_text_with_code-checkpoint.txt

paper_id=2 source=review_outputs/paper_reviews/paper_002/article_extracted_reader_text_with_code.txt
   - article_extracted_reader_text_with_code.txt
   - article_extracted_reader_text_with_code-checkpoint.txt

paper_id=3 source=review_outputs/paper_reviews/paper_003/article_extracted_reader_text_with_code.txt
   - article_extracted_reader_text_with_code.txt
   - article_extracted_reader_text_with_code-checkpoint.txt

Preparing Grid'5000 HOME...
[CLEANUP] removing old launcher/model/cache artifacts from HOME and /tmp
[CLEANUP] done
[CLEANUP] HOME model-store:
16K	/home/.../model-store
[CLEANUP] filesystem usage:
Filesystem        Size  Used Avail Use% Mounted on
nfs:/export/home   15T  2.2T   12T  16% /home
/dev/sd

In [4]:
# from pathlib import Path
# import json

# job_dir = max(Path("g5k_paper_chunk_results").glob("job_*"), key=lambda p: p.stat().st_mtime)
# print("JOB DIR:", job_dir)

# summary_path = job_dir / "summary.json"
# print("\nSUMMARY PATH:", summary_path)
# print(summary_path.read_text(encoding="utf-8"))

In [5]:
from pathlib import Path

job_root = Path("./g5k_paper_chunk_results")
job_dirs = sorted(job_root.glob("job_*"), key=lambda p: p.stat().st_mtime)

print("latest job dir:", job_dirs[-1] if job_dirs else "NONE")

if job_dirs:
    latest = job_dirs[-1]
    remote_copy = latest / "outputs" / "paper_collections"
    print("downloaded outputs exists:", remote_copy.exists())
    if remote_copy.exists():
        for p in sorted(remote_copy.rglob("*")):
            print(p)

latest job dir: g5k_paper_chunk_results/job_ ... _20260501_100618
downloaded outputs exists: True
g5k_paper_chunk_results/job_ ... _20260501_100618/outputs/paper_collections/paper_1
g5k_paper_chunk_results/job_ ... _20260501_100618/outputs/paper_collections/paper_1/chunks.jsonl
g5k_paper_chunk_results/job_ ... _20260501_100618/outputs/paper_collections/paper_1/manifest.json
g5k_paper_chunk_results/job_ ... _20260501_100618/outputs/paper_collections/paper_2
g5k_paper_chunk_results/job_ ... _20260501_100618/outputs/paper_collections/paper_2/chunks.jsonl
g5k_paper_chunk_results/job_ ... _20260501_100618/outputs/paper_collections/paper_2/manifest.json
g5k_paper_chunk_results/job_ ... _20260501_100618/outputs/paper_collections/paper_3
g5k_paper_chunk_results/job_ ... _20260501_100618/outputs/paper_collections/paper_3/chunks.jsonl
g5k_paper_chunk_results/job_ ... _20260501_100618/outputs/paper_collections/paper_3/manifest.json
